In [550]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [551]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches
from src.elo import prepare_matches, update_elo
from src.simulation import predict_match, simulate_match, simulate_group, simulate_group_stage, simulate_knockout_match, construct_round32, simulate_knockout_round, construct_round16, construct_QF, construct_SF, simulate_tournament
from src.features import update_team_history, update_h2h, build_features, assign_third_place_slots
import numpy as np
from collections import Counter
from src.simulation import run_tournament

#Load everything from earlier notebooks

history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')
model_h = joblib.load('home_goals_model.joblib')
model_a = joblib.load('away_goals_model.joblib')
features = joblib.load('model_features.joblib')

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")
df_confederations = pd.read_csv('../data/FIFA_confederations.csv')

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

In [552]:
team_to_group = {team: group for group, teams in wc_groups.items() for team in teams}
team_to_confederation = dict(zip(df_confederations["nation"], df_confederations["confederation"]))

# Create group_stages equivalent
rows = []
for group, teams in wc_groups.items():
    for i, team in enumerate(teams, 1):
        rows.append({"group": group, "position": i, "nation": team})
df_groups = pd.DataFrame(rows)

# Create fixtures equivalent — you already have this
df_group_fixtures = group_stage_matches.copy()
df_group_fixtures["group"] = df_group_fixtures["home_team"].map(team_to_group)

In [553]:
#Example run
x=build_features('Argentina', 'Brazil', country_elo, team_to_confederation, features)
x


,const,home_elo,away_elo,neutral,tournament_weight,home_confederation_AFC,home_confederation_CAF,home_confederation_CONCACAF,home_confederation_CONMEBOL,home_confederation_OFC,home_confederation_UEFA,home_confederation_Unknown,away_confederation_AFC,away_confederation_CAF,away_confederation_CONCACAF,away_confederation_CONMEBOL,away_confederation_OFC,away_confederation_UEFA,away_confederation_Unknown
0,1.0,2113.427162,1989.655735,1.0,5.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [554]:
y = simulate_match('Argentina', 'Brazil', model_h, model_a, country_elo, team_to_confederation, features)
y

{'home_team': 'Argentina',
 'away_team': 'Brazil',
 'home_goals': 0,
 'away_goals': 0,
 'result': 'draw',
 'winner': None}

In [555]:
group_a_table, group_a_matches = simulate_group("C", df_groups, df_group_fixtures, model_h, model_a, country_elo, team_to_confederation, features)

display(group_a_matches)
display(group_a_table)

,home_team,away_team,home_goals,away_goals,result,winner
0,Brazil,Morocco,2,1,home_win,Brazil
1,Haiti,Scotland,1,1,draw,NaN
2,Scotland,Morocco,0,0,draw,NaN
3,Brazil,Haiti,4,0,home_win,Brazil
4,Scotland,Brazil,0,1,away_win,Brazil
5,Morocco,Haiti,1,0,home_win,Morocco


,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,Brazil,3,3,0,0,7,1,6,9,1,C
1,Morocco,3,1,1,1,2,2,0,4,2,C
2,Scotland,3,0,2,1,1,2,-1,2,3,C
3,Haiti,3,0,1,2,1,6,-5,1,4,C


In [556]:
group_stage = simulate_group_stage(df_groups, df_group_fixtures, model_h, model_a, country_elo, team_to_confederation, features)
group_stage_result = group_stage[0]

top2 = group_stage_result.groupby('group').head(2).copy() #Teams that placed top 2 in their group which qualifies
third = group_stage_result.groupby('group').nth(2).copy() #All teams that placed third (only 8 of them move on)
best8_third = third.sort_values(
    ['points', 'goal_difference', 'goals_for'], 
    ascending=[False, False, False]
).head(8).reset_index(drop=True)

best8_third['third_place_rank'] = best8_third.index + 1

display(group_stage_result)

,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,Mexico,3,3,0,0,5,0,5,9,1,A
1,South Korea,3,2,0,1,5,2,3,6,2,A
2,South Africa,3,1,0,2,3,2,1,3,3,A
3,Czech Republic,3,0,0,3,1,10,-9,0,4,A
4,Switzerland,3,2,0,1,4,3,1,6,1,B
5,Canada,3,1,1,1,4,3,1,4,2,B
6,Bosnia and Herzegovina,3,1,1,1,2,2,0,4,3,B
7,Qatar,3,0,2,1,2,4,-2,2,4,B
8,Brazil,3,3,0,0,5,1,4,9,1,C
9,Morocco,3,2,0,1,3,3,0,6,2,C


In [557]:
round_of_32 = pd.concat([top2, best8_third])
round_of_32

,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group,third_place_rank
0,Mexico,3,3,0,0,5,0,5,9,1,A,NaN
1,South Korea,3,2,0,1,5,2,3,6,2,A,NaN
4,Switzerland,3,2,0,1,4,3,1,6,1,B,NaN
5,Canada,3,1,1,1,4,3,1,4,2,B,NaN
8,Brazil,3,3,0,0,5,1,4,9,1,C,NaN
9,Morocco,3,2,0,1,3,3,0,6,2,C,NaN
12,Paraguay,3,2,1,0,4,2,2,7,1,D,NaN
13,United States,3,1,1,1,3,3,0,4,2,D,NaN
16,Germany,3,3,0,0,7,1,6,9,1,E,NaN
17,Ecuador,3,2,0,1,8,4,4,6,2,E,NaN


In [558]:
rd32_teams = {}
for team in top2.itertuples(index=False):
        rd32_teams[f'{team.group_rank}{team.group}'] = team.team
        
thirds = assign_third_place_slots(best8_third)
rd32_teams |= thirds

x = construct_round32(rd32_teams)
print(rd32_teams)
simulate_knockout_match(x[73][0], x[73][1], model_h, model_a, country_elo, team_to_confederation, features)

{'1A': 'Mexico', '2A': 'South Korea', '1B': 'Switzerland', '2B': 'Canada', '1C': 'Brazil', '2C': 'Morocco', '1D': 'Paraguay', '2D': 'United States', '1E': 'Germany', '2E': 'Ecuador', '1F': 'Sweden', '2F': 'Netherlands', '1G': 'Iran', '2G': 'Belgium', '1H': 'Spain', '2H': 'Uruguay', '1I': 'Senegal', '2I': 'France', '1J': 'Argentina', '2J': 'Jordan', '1K': 'Portugal', '2K': 'Colombia', '1L': 'Croatia', '2L': 'Panama', '3ABCDF': 'Bosnia and Herzegovina', '3CDFGH': 'Turkey', '3CEFHI': 'Norway', '3EHIJK': 'Austria', '3BEFIJ': 'Ivory Coast', '3AEHIJ': 'South Africa', '3EFGIJ': 'New Zealand', '3DEIJL': 'England'}


{'home_team': 'South Korea',
 'away_team': 'Canada',
 'home_goals': 1,
 'away_goals': 1,
 'result': 'home_win',
 'result_type': 'OT/Pens',
 'winner': 'South Korea',
 'loser': 'Canada',
 'home_win_prob': np.float64(0.34063814709338164),
 'draw_prob': np.float64(0.27658444942514965),
 'away_win_prob': np.float64(0.38276748107801073)}

In [559]:
winners, results = simulate_knockout_round(x, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,South Korea,Canada,0,0,OT/Pens,Canada,South Korea,0.340638,0.276584,0.382767
1,Germany,Bosnia and Herzegovina,0,1,regular_time,Bosnia and Herzegovina,Germany,0.685382,0.200070,0.114328
2,Sweden,Morocco,2,1,regular_time,Sweden,Morocco,0.164946,0.248638,0.586371
3,Brazil,Netherlands,1,0,regular_time,Brazil,Netherlands,0.452546,0.289080,0.258367
4,Senegal,Turkey,1,0,regular_time,Senegal,Turkey,0.297184,0.302026,0.400786
5,Germany,France,2,2,OT/Pens,France,Germany,0.182703,0.252067,0.565189
6,Mexico,Norway,2,1,regular_time,Mexico,Norway,0.425989,0.285293,0.288710
7,Croatia,Austria,1,3,regular_time,Austria,Croatia,0.390163,0.286116,0.323714
8,Paraguay,Ivory Coast,1,0,regular_time,Paraguay,Ivory Coast,0.552046,0.267958,0.179974
9,Iran,South Africa,1,0,regular_time,Iran,South Africa,0.697711,0.201687,0.100427


In [560]:
r16_matchups = construct_round16(winners)

winners, results = simulate_knockout_round(r16_matchups, model_h, model_a, country_elo, team_to_confederation, features)

display(winners)

{89: 'Senegal',
 90: 'Canada',
 91: 'Brazil',
 92: 'Austria',
 93: 'Spain',
 94: 'Iran',
 95: 'Argentina',
 96: 'England'}

In [561]:
QF_matchsups = construct_QF(winners)

winners, results = simulate_knockout_round(QF_matchsups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Senegal,Canada,1,0,regular_time,Senegal,Canada,0.358829,0.291275,0.349891
1,Spain,Iran,1,0,regular_time,Spain,Iran,0.635599,0.233935,0.130399
2,Brazil,Austria,2,4,regular_time,Austria,Brazil,0.583032,0.250704,0.166222
3,Argentina,England,2,0,regular_time,Argentina,England,0.494938,0.283704,0.221347


In [562]:
SF_matchsups = construct_SF(winners)

winners, results = simulate_knockout_round(SF_matchsups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Senegal,Spain,0,5,regular_time,Spain,Senegal,0.111484,0.219311,0.669104
1,Austria,Argentina,1,4,regular_time,Argentina,Austria,0.074048,0.158404,0.766901


In [563]:
winner, results = simulate_knockout_round({103: (winners[101], winners[102])}, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

print(f'{winner[103]} won the World Cup')

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Spain,Argentina,0,0,OT/Pens,Argentina,Spain,0.29184,0.291956,0.416199


Argentina won the World Cup


In [ ]:
simulate_tournament(model_h, model_a, country_elo, team_to_confederation, features, df_groups, df_group_fixtures)